In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import shutil
import glob

DRIVE_DIR = "/content/drive/MyDrive/Processed_SP_ML"
LOCAL_DIR = "/content/Processed_SP_ML"
RESULTS_DIR = "/content/drive/MyDrive/CNN_results"

# Copy dataset from Google Drive to Colab
if not os.path.exists(LOCAL_DIR):
    shutil.copytree(DRIVE_DIR, LOCAL_DIR)

# Create results directory
os.makedirs(RESULTS_DIR, exist_ok=True)

# Count images
IMAGE_DIR = os.path.join(LOCAL_DIR, "images")
print("Images:", len(os.listdir(IMAGE_DIR)))

# Find severity labels Excel file
severity_matches = glob.glob(
    "/content/drive/MyDrive/**/Severity_Labels_PerDisease.xlsx",
    recursive=True
)

if not severity_matches:
    raise FileNotFoundError(
        "Could not find Severity_Labels_PerDisease.xlsx in Google Drive."
    )

SEV_FILE = severity_matches[0]
print("Severity file:", SEV_FILE)

Images: 1032
Severity file: /content/drive/MyDrive/Severity_Labels_PerDisease.xlsx


In [3]:
import json, time, copy, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, recall_score, f1_score, confusion_matrix
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "NONE - turn on T4 GPU!")

device: cuda | GPU: Tesla T4


In [4]:
BACKBONE = "densenet121"
PRETRAINED = True
IMG_W, IMG_H = 384, 288
EPOCHS, WARMUP_EPOCHS, PATIENCE = 30, 2, 7
BATCH_SIZE = 16
LR_HEAD, LR_BACKBONE, WEIGHT_DECAY = 1e-3, 1e-4, 1e-4
SEED = 42
DIS_COLS = ["FECD", "Guttata", "Folds", "PolyPleo", "LowCD"]
SEV_HEADS = {"FECDgroup": ["FECD", "PolyPleo", "LowCD"], "Folds": ["Folds"]}   # severity outputs
SEV_COLS = list(SEV_HEADS)
FEATURES = ["ECD_CD_cells_per_mm2", "ECD_Deficit", "SD", "CV", "HEX_percent", "Donor_Age"]
RUN_DIR = os.path.join(RESULTS_DIR, f"severity_{BACKBONE}")
os.makedirs(RUN_DIR, exist_ok=True)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

man = pd.read_csv(os.path.join(LOCAL_DIR, "Image_Manifest_MultiLabel.csv"))
sev = pd.read_excel(SEV_FILE, sheet_name="Severity_Labels", keep_default_na=False, na_values=[""])
sev = sev[["Tissue_ID", "FECDgroup_Sev", "Folds_Sev", "Guttata_Sev", "Overall_Sev"]]
df = man.merge(sev, on="Tissue_ID", how="left")
print("Rows without severity info:", df["FECDgroup_Sev"].isna().sum())

# severity target: 1 = Moderate/Severe, 0 = Mild ; mask = disease present AND severity known
for h in SEV_COLS:
    s = df[f"{h}_Sev"].astype(str)
    df[f"SevY_{h}"] = s.isin(["Moderate", "Severe"]).astype(float)
    df[f"SevM_{h}"] = s.isin(["Mild", "Moderate", "Severe"]).astype(float)
df["Label_Origin"] = np.where(df["Tissue_ID"].str.contains("NEB-2023"), "Model-assisted (2023)", "Original")

train_df = df[df["Split"] == "train"].reset_index(drop=True)
test_df  = df[df["Split"] == "test"].reset_index(drop=True)
N_FOLDS = int(train_df["CV_Fold"].max()) + 1
summary = pd.DataFrame({f"{h} {p}": [int(d[f"SevM_{h}"].sum() - d[f"SevY_{h}"].sum()) if p == "Mild" else int(d[f"SevY_{h}"].sum())
                                      for d in (train_df, test_df)] for h in SEV_COLS for p in ("Mild", "Mod+Sev")},
                       index=["Train", "Test"])
print(f"{BACKBONE} | train {len(train_df)} | test {len(test_df)} | folds {N_FOLDS}")
summary

Rows without severity info: 0
densenet121 | train 824 | test 208 | folds 5


,FECDgroup Mild,FECDgroup Mod+Sev,Folds Mild,Folds Mod+Sev
Train,93,13,268,35
Test,20,7,67,6


In [5]:
IMNET_MEAN, IMNET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
T = torchvision.transforms
TF_TRAIN = T.Compose([T.ToPILImage(),
                      T.RandomResizedCrop((IMG_H, IMG_W), scale=(0.7, 1.0), ratio=(1.25, 1.42)),
                      T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(15),
                      T.ColorJitter(brightness=0.2, contrast=0.2),
                      T.Grayscale(3), T.ToTensor(), T.Normalize(IMNET_MEAN, IMNET_STD)])
TF_EVAL = T.Compose([T.ToPILImage(), T.Resize((IMG_H, IMG_W)),
                     T.Grayscale(3), T.ToTensor(), T.Normalize(IMNET_MEAN, IMNET_STD)])

def fit_tab_scaler(frame):
    imp = SimpleImputer(strategy="median").fit(frame[FEATURES])
    sc = StandardScaler().fit(imp.transform(frame[FEATURES]))
    return lambda f: sc.transform(imp.transform(f[FEATURES])).astype(np.float32)

class SPDataset(Dataset):
    def __init__(self, frame, tab, train):
        self.paths = [os.path.join(LOCAL_DIR, "images", f) for f in frame["Proc_File"]]
        self.Yd = frame[DIS_COLS].values.astype(np.float32)
        self.Ys = frame[[f"SevY_{h}" for h in SEV_COLS]].values.astype(np.float32)
        self.Ms = frame[[f"SevM_{h}" for h in SEV_COLS]].values.astype(np.float32)
        self.tab = torch.tensor(tab, dtype=torch.float32)
        self.tf = TF_TRAIN if train else TF_EVAL
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        x = self.tf(cv2.imread(self.paths[i], cv2.IMREAD_GRAYSCALE))
        return x, self.tab[i], torch.tensor(self.Yd[i]), torch.tensor(self.Ys[i]), torch.tensor(self.Ms[i])

class FusionMultiTask(nn.Module):
    """Image (DenseNet121) + 6 numbers -> shared features -> 5 disease outputs + 2 severity outputs."""
    def __init__(self):
        super().__init__()
        m = models.densenet121(weights="DEFAULT" if PRETRAINED else None)
        dim = m.classifier.in_features; m.classifier = nn.Identity()
        self.backbone, self.cam_layer = m, m.features
        self.tab_mlp = nn.Sequential(nn.Linear(len(FEATURES), 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 32), nn.ReLU())
        self.shared = nn.Sequential(nn.Dropout(0.3), nn.Linear(dim + 32, 128), nn.ReLU(), nn.Dropout(0.3))
        self.disease_head = nn.Linear(128, len(DIS_COLS))
        self.severity_head = nn.Linear(128, len(SEV_COLS))
    def forward(self, x, tab):
        z = self.shared(torch.cat([self.backbone(x), self.tab_mlp(tab)], 1))
        return self.disease_head(z), self.severity_head(z)

print("OK - multi-task model defined (5 disease + 2 severity outputs)")

OK - multi-task model defined (5 disease + 2 severity outputs)


In [6]:
def pos_weight(Y, M=None):
    M = np.ones_like(Y) if M is None else M
    pos = (Y * M).sum(0); neg = M.sum(0) - pos
    return torch.tensor(np.clip(neg / np.maximum(pos, 1), 1, 10), dtype=torch.float32, device=DEVICE)

def make_loader(frame, scale, train):
    return DataLoader(SPDataset(frame, scale(frame), train), batch_size=BATCH_SIZE if train else 32,
                      shuffle=train, drop_last=train, num_workers=2, pin_memory=True)

@torch.no_grad()
def predict(model, dl):
    model.eval(); PD, PS = [], []
    for x, tab, *_ in dl:
        x, tab = x.to(DEVICE), tab.to(DEVICE)
        with torch.autocast("cuda", enabled=DEVICE == "cuda"):
            d1, s1 = model(x, tab); d2, s2 = model(torch.flip(x, dims=[3]), tab)
        PD.append(((torch.sigmoid(d1) + torch.sigmoid(d2)) / 2).float().cpu().numpy())
        PS.append(((torch.sigmoid(s1) + torch.sigmoid(s2)) / 2).float().cpu().numpy())
    return np.concatenate(PD), np.concatenate(PS)

def safe_auc(y, p):
    return roc_auc_score(y, p) if 0 < y.sum() < len(y) else np.nan

def val_score(frame, PD, PS):
    d = np.nanmean([safe_auc(frame[c].values, PD[:, j]) for j, c in enumerate(DIS_COLS)])
    s = []
    for j, h in enumerate(SEV_COLS):
        m = frame[f"SevM_{h}"].values == 1
        s.append(safe_auc(frame.loc[m, f"SevY_{h}"].values, PS[m, j]))
    return 0.5 * d + 0.5 * np.nanmean(s), d, s

def train_one(tr_frame, va_frame=None, epochs=EPOCHS, log=True):
    set_seed()
    scale = fit_tab_scaler(tr_frame)
    tr_dl = make_loader(tr_frame, scale, True)
    va_dl = None if va_frame is None else make_loader(va_frame, scale, False)
    model = FusionMultiTask().to(DEVICE)
    head = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    crit_d = nn.BCEWithLogitsLoss(pos_weight=pos_weight(tr_frame[DIS_COLS].values))
    crit_s = nn.BCEWithLogitsLoss(pos_weight=pos_weight(tr_frame[[f"SevY_{h}" for h in SEV_COLS]].values,
                                                        tr_frame[[f"SevM_{h}" for h in SEV_COLS]].values), reduction="none")
    opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": LR_BACKBONE},
                             {"params": head, "lr": LR_HEAD}], weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs - WARMUP_EPOCHS, 1))
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")
    best, best_sc, best_ep, wait = None, -1, epochs, 0
    for ep in range(1, epochs + 1):
        warm = ep <= WARMUP_EPOCHS
        for p in model.backbone.parameters(): p.requires_grad = not warm
        model.train(); t0, losses = time.time(), []
        for x, tab, yd, ys, ms in tr_dl:
            x, tab, yd, ys, ms = [t.to(DEVICE) for t in (x, tab, yd, ys, ms)]
            opt.zero_grad()
            with torch.autocast("cuda", enabled=DEVICE == "cuda"):
                ld, ls = model(x, tab)
                loss_d = crit_d(ld, yd)
                loss_s = (crit_s(ls.float(), ys) * ms).sum() / ms.sum().clamp(min=1)   # masked: only known severities
                loss = loss_d + loss_s
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); losses.append(loss.item())
        if not warm: sched.step()
        msg = f"  ep {ep:2d} loss {np.mean(losses):.4f}"
        if va_dl is not None:
            sc, d, s = val_score(va_frame, *predict(model, va_dl))
            msg += f" | val disease AUC {d:.3f} | val severity AUC " + " / ".join(f"{v:.3f}" for v in s)
            if sc > best_sc: best_sc, best_ep, best, wait = sc, ep, copy.deepcopy(model.state_dict()), 0
            elif not warm: wait += 1
        if log: print(msg + (" (warm-up)" if warm else "") + f" ({time.time() - t0:.0f}s)")
        if va_dl is not None and wait >= PATIENCE: break
    if best is not None: model.load_state_dict(best)
    for p in model.parameters(): p.requires_grad = True
    return model, scale, best_ep

print("OK - training functions ready")

OK - training functions ready


In [7]:
oof_path, info_path = os.path.join(RUN_DIR, "oof_predictions.csv"), os.path.join(RUN_DIR, "fold_info.json")
pcols = [f"P_{c}" for c in DIS_COLS] + [f"PSev_{h}" for h in SEV_COLS]
oof = pd.read_csv(oof_path) if os.path.exists(oof_path) else train_df[["Tissue_ID", "CV_Fold"]].assign(**{c: np.nan for c in pcols})
folds = json.load(open(info_path)) if os.path.exists(info_path) else {}

for k in range(N_FOLDS):
    if str(k) in folds:
        print(f"Fold {k} already done - skipping"); continue
    tr_f, va_f = train_df[train_df["CV_Fold"] != k], train_df[train_df["CV_Fold"] == k]
    print(f"\n===== Severity multi-task | fold {k} | train {len(tr_f)} / val {len(va_f)} =====")
    t0 = time.time()
    model, scale, best_ep = train_one(tr_f, va_f)
    PD, PS = predict(model, make_loader(va_f, scale, False))
    oof.loc[oof["Tissue_ID"].isin(va_f["Tissue_ID"]), pcols] = np.hstack([PD, PS])
    oof.to_csv(oof_path, index=False)
    sc, d, s = val_score(va_f, PD, PS)
    folds[str(k)] = {"best_epoch": int(best_ep), "disease_auc": float(d), "severity_auc": [float(v) for v in s],
                     "minutes": round((time.time() - t0) / 60, 1)}
    json.dump(folds, open(info_path, "w"), indent=2)
    print(f"Fold {k}: disease AUC {d:.3f} | severity AUC {s} | best epoch {best_ep} - SAVED to Drive")
    del model; torch.cuda.empty_cache()


===== Severity multi-task | fold 0 | train 660 / val 164 =====
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 126MB/s]


  ep  1 loss 2.4819 | val disease AUC 0.651 | val severity AUC 0.975 / 0.639 (warm-up) (12s)
  ep  2 loss 2.3502 | val disease AUC 0.739 | val severity AUC 0.950 / 0.804 (warm-up) (13s)
  ep  3 loss 2.4591 | val disease AUC 0.807 | val severity AUC 1.000 / 0.968 (17s)
  ep  4 loss 2.2507 | val disease AUC 0.791 | val severity AUC 0.975 / 0.981 (14s)
  ep  5 loss 1.9946 | val disease AUC 0.842 | val severity AUC 0.850 / 1.000 (16s)
  ep  6 loss 1.8278 | val disease AUC 0.882 | val severity AUC 1.000 / 0.997 (15s)
  ep  7 loss 1.6597 | val disease AUC 0.860 | val severity AUC 1.000 / 0.889 (10s)
  ep  8 loss 1.5893 | val disease AUC 0.891 | val severity AUC 1.000 / 0.991 (10s)
  ep  9 loss 1.4729 | val disease AUC 0.886 | val severity AUC 1.000 / 0.988 (10s)
  ep 10 loss 1.4007 | val disease AUC 0.898 | val severity AUC 1.000 / 0.981 (10s)
  ep 11 loss 1.2628 | val disease AUC 0.898 | val severity AUC 1.000 / 0.977 (11s)
  ep 12 loss 1.3114 | val disease AUC 0.913 | val severity AUC 1.00

In [8]:
o = train_df.merge(oof[["Tissue_ID"] + pcols], on="Tissue_ID")
rf = lambda: RandomForestClassifier(n_estimators=400, min_samples_leaf=3, class_weight="balanced_subsample",
                                    random_state=SEED, n_jobs=-1)
rows, THR = [], {}
for j, h in enumerate(SEV_COLS):
    m = o[f"SevM_{h}"] == 1
    y, p = o.loc[m, f"SevY_{h}"].values, o.loc[m, f"PSev_{h}"].values
    num_p = np.zeros(m.sum())                                  # numbers-only baseline on the SAME folds
    sub = o[m].reset_index(drop=True)
    for k in range(N_FOLDS):
        tr, va = sub["CV_Fold"] != k, sub["CV_Fold"] == k
        if va.sum() == 0 or sub.loc[tr, f"SevY_{h}"].nunique() < 2: continue
        sc_ = fit_tab_scaler(sub[tr]); mdl = rf().fit(sc_(sub[tr]), sub.loc[tr, f"SevY_{h}"])
        num_p[va.values] = mdl.predict_proba(sc_(sub[va]))[:, 1]
    fpr, tpr, thr = roc_curve(y, p); b = np.argmax(tpr - fpr); THR[h] = float(np.clip(thr[b], 0.01, 0.99))
    rows.append({"Severity output": h, "n (Mild / Mod+Sev)": f"{int((y==0).sum())} / {int(y.sum())}",
                 "Numbers only AUC": safe_auc(y, num_p), "Image + numbers AUC": safe_auc(y, p),
                 "Sensitivity": tpr[b], "Specificity": 1 - fpr[b], "Threshold": THR[h]})
cv_sev = pd.DataFrame(rows).set_index("Severity output")
cv_dis = pd.Series({c: safe_auc(o[c].values, o[f"P_{c}"].values) for c in DIS_COLS}, name="CV disease AUC")
DTHR = {}
for c in DIS_COLS:
    fpr, tpr, thr = roc_curve(o[c], o[f"P_{c}"]); DTHR[c] = float(np.clip(thr[np.argmax(tpr - fpr)], 0.01, 0.99))
cv_sev.to_csv(os.path.join(RUN_DIR, "cv_severity.csv"))
print("TRAIN CV - severity (Mild vs Moderate/Severe), only corneas that HAVE the disease")
display(cv_sev.round(3))
print("TRAIN CV - disease detection of the same multi-task model (should be close to notebook 06a)")
display(cv_dis.round(3).to_frame().T)

TRAIN CV - severity (Mild vs Moderate/Severe), only corneas that HAVE the disease


,n (Mild / Mod+Sev),Numbers only AUC,Image + numbers AUC,Sensitivity,Specificity,Threshold
Severity output,,,,,,
FECDgroup,93 / 13,0.939,0.880,0.923,0.849,0.434
Folds,268 / 35,0.503,0.857,0.914,0.687,0.277


TRAIN CV - disease detection of the same multi-task model (should be close to notebook 06a)


,FECD,Guttata,Folds,PolyPleo,LowCD
CV disease AUC,0.973,0.818,0.788,0.972,0.977


In [9]:
ep = max(int(np.median([v["best_epoch"] for v in folds.values()])), WARMUP_EPOCHS + 1)
print(f"Final multi-task model: {ep} epochs on all {len(train_df)} train images")
final_model, final_scale, _ = train_one(train_df, None, epochs=ep)
PD_TE, PS_TE = predict(final_model, make_loader(test_df, final_scale, False))
torch.save(final_model.state_dict(), os.path.join(RUN_DIR, "final_multitask.pt"))
t = test_df.copy()
for j, c in enumerate(DIS_COLS): t[f"P_{c}"] = PD_TE[:, j]
for j, h in enumerate(SEV_COLS): t[f"PSev_{h}"] = PS_TE[:, j]
print("OK - test predictions ready")

Final multi-task model: 14 epochs on all 824 train images
  ep  1 loss 2.4981 (warm-up) (7s)
  ep  2 loss 2.4272 (warm-up) (11s)
  ep  3 loss 2.2479 (19s)
  ep  4 loss 2.0126 (11s)
  ep  5 loss 1.6869 (10s)
  ep  6 loss 1.5178 (10s)
  ep  7 loss 1.5153 (12s)
  ep  8 loss 1.4483 (11s)
  ep  9 loss 1.3592 (10s)
  ep 10 loss 1.2295 (11s)
  ep 11 loss 1.1174 (10s)
  ep 12 loss 1.2992 (10s)
  ep 13 loss 1.1088 (11s)
  ep 14 loss 0.9936 (10s)
OK - test predictions ready


In [10]:
rng = np.random.default_rng(SEED)
def auc_ci(y, p, n=1000):
    s = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(i): s.append(roc_auc_score(y[i], p[i]))
    return (np.percentile(s, 2.5), np.percentile(s, 97.5)) if s else (np.nan, np.nan)

rows = []
for h in SEV_COLS:
    trm = train_df[f"SevM_{h}"] == 1
    sc_ = fit_tab_scaler(train_df[trm]); mdl = rf().fit(sc_(train_df[trm]), train_df.loc[trm, f"SevY_{h}"])
    m = t[f"SevM_{h}"] == 1
    y, p = t.loc[m, f"SevY_{h}"].values, t.loc[m, f"PSev_{h}"].values
    pn = mdl.predict_proba(sc_(t[m]))[:, 1]
    yhat = (p >= THR[h]).astype(int)
    lo, hi = auc_ci(y, p)
    rows.append({"Severity output": h, "n (Mild / Mod+Sev)": f"{int((y==0).sum())} / {int(y.sum())}",
                 "Numbers only AUC": safe_auc(y, pn), "Image + numbers AUC": safe_auc(y, p),
                 "95% CI": f"[{lo:.2f}-{hi:.2f}]", "Sensitivity": recall_score(y, yhat, zero_division=0),
                 "Specificity": recall_score(y, yhat, pos_label=0, zero_division=0)})
test_sev = pd.DataFrame(rows).set_index("Severity output")
test_sev.to_csv(os.path.join(RUN_DIR, "test_severity.csv"))
print("HELD-OUT TEST - severity (Mild vs Moderate/Severe), corneas that truly have the disease")
display(test_sev.round(3))

rows = []
for h in SEV_COLS:
    for org in ["Original", "Model-assisted (2023)"]:
        m = (t[f"SevM_{h}"] == 1) & (t["Label_Origin"] == org)
        y, p = t.loc[m, f"SevY_{h}"].values, t.loc[m, f"PSev_{h}"].values
        rows.append({"Severity output": h, "Label origin": org, "n (Mild / Mod+Sev)": f"{int((y==0).sum())} / {int(y.sum())}",
                     "AUC": safe_auc(y, p)})
print("Test severity AUC by label origin (NaN = not enough cases of both classes)")
display(pd.DataFrame(rows).round(3))

HELD-OUT TEST - severity (Mild vs Moderate/Severe), corneas that truly have the disease


,n (Mild / Mod+Sev),Numbers only AUC,Image + numbers AUC,95% CI,Sensitivity,Specificity
Severity output,,,,,,
FECDgroup,20 / 7,1.000,1.000,[1.00-1.00],1.000,1.000
Folds,67 / 6,0.408,0.766,[0.57-0.94],0.833,0.612


Test severity AUC by label origin (NaN = not enough cases of both classes)


,Severity output,Label origin,n (Mild / Mod+Sev),AUC
0,FECDgroup,Original,6 / 4,1.000
1,FECDgroup,Model-assisted (2023),14 / 3,1.000
2,Folds,Original,36 / 2,1.000
3,Folds,Model-assisted (2023),31 / 4,0.726


In [11]:
ORD = {"None": 0, "Mild": 1, "Mod/Sev": 2}
def cornea_report(r):
    found, sevs = [], []
    for c in DIS_COLS:
        if r[f"P_{c}"] >= DTHR[c]: found.append(c)
    if any(c in found for c in SEV_HEADS["FECDgroup"]):
        s = "Mod/Sev" if r["PSev_FECDgroup"] >= THR["FECDgroup"] else "Mild"; sevs.append(("FECD group", s))
    if "Folds" in found:
        s = "Mod/Sev" if r["PSev_Folds"] >= THR["Folds"] else "Mild"; sevs.append(("Folds", s))
    if "Guttata" in found:
        sevs.append(("Guttata", "Mild"))          # not gradable: all recorded guttata severities are mild
    overall = max([s for _, s in sevs], key=lambda s: ORD[s]) if sevs else "None"
    return pd.Series({"Predicted diseases": " + ".join(found) if found else "Normal",
                      "Predicted severity": ", ".join(f"{d}: {s}" for d, s in sevs) if sevs else "-",
                      "Predicted overall": overall})
rep = pd.concat([t[["Tissue_ID", "Combination", "Overall_Sev"]], t.apply(cornea_report, axis=1)], axis=1)
rep["True overall"] = rep["Overall_Sev"].astype(str).replace({"Moderate": "Mod/Sev", "Severe": "Mod/Sev", "?": "Mod/Sev", "nan": "None"})
labels = ["None", "Mild", "Mod/Sev"]
cm = pd.DataFrame(confusion_matrix(rep["True overall"], rep["Predicted overall"], labels=labels),
                  index=[f"True {l}" for l in labels], columns=[f"Pred {l}" for l in labels])
print("End-to-end overall cornea severity (test): true vs predicted")
display(cm)
print(f"Overall severity exact agreement: {(rep['True overall'] == rep['Predicted overall']).mean():.1%}")
print("\nExample reports (test):")
display(rep.drop(columns="Overall_Sev").sample(12, random_state=1).reset_index(drop=True))

End-to-end overall cornea severity (test): true vs predicted


,Pred None,Pred Mild,Pred Mod/Sev
True None,72,21,6
True Mild,18,47,29
True Mod/Sev,0,3,12


Overall severity exact agreement: 63.0%

Example reports (test):


,Tissue_ID,Combination,Predicted diseases,Predicted severity,Predicted overall,True overall
0,NEB-2025-10-17541 L,Normal,Normal,-,None,None
1,NEB-2024-08-16268 R,Folds,Guttata + Folds,"Folds: Mod/Sev, Guttata: Mild",Mod/Sev,Mild
2,NEB-2024-08-16294 L,Folds,Folds,Folds: Mod/Sev,Mod/Sev,Mild
3,NEB-2025-11-17629 R,Normal,Normal,-,None,None
4,NEB-2023-03-14810 R,FECD + PolyPleo + LowCD,FECD + PolyPleo + LowCD,FECD group: Mild,Mild,Mild
5,NEB-2020-12-12850 L,Folds,Folds,Folds: Mild,Mild,Mild
6,NEB-2024-08-16260 L,Folds,Normal,-,None,Mild
7,NEB-2020-10-12747 L,Guttata + Folds,Guttata + Folds,"Folds: Mod/Sev, Guttata: Mild",Mod/Sev,Mild
8,NEB-2025-11-17633 R,Folds,Guttata + Folds,"Folds: Mod/Sev, Guttata: Mild",Mod/Sev,Mild
9,NEB-2023-03-14830 R,FECD + PolyPleo + LowCD,FECD + PolyPleo + LowCD,FECD group: Mild,Mild,Mild


In [12]:
rep.drop(columns="Overall_Sev").to_csv(os.path.join(RUN_DIR, "test_cornea_reports.csv"), index=False)
t[["Tissue_ID", "Label_Origin"] + DIS_COLS + [f"SevY_{h}" for h in SEV_COLS] + [f"SevM_{h}" for h in SEV_COLS]
  + [f"P_{c}" for c in DIS_COLS] + [f"PSev_{h}" for h in SEV_COLS]].to_csv(os.path.join(RUN_DIR, "test_predictions_severity.csv"), index=False)
json.dump({"backbone": BACKBONE, "severity_heads": SEV_HEADS, "severity_thresholds": THR, "disease_thresholds": DTHR,
           "cv_severity": cv_sev.round(4).to_dict(orient="index"), "test_severity": test_sev.round(4).to_dict(orient="index"),
           "cv_disease_auc": cv_dis.round(4).to_dict(), "epochs_final": ep, "folds": folds},
          open(os.path.join(RUN_DIR, "summary_severity.json"), "w"), indent=2, default=str)
print("Saved to", RUN_DIR); print(os.listdir(RUN_DIR))

Saved to /content/drive/MyDrive/CNN_results/severity_densenet121
['oof_predictions.csv', 'fold_info.json', 'cv_severity.csv', 'final_multitask.pt', 'test_severity.csv', 'test_cornea_reports.csv', 'test_predictions_severity.csv', 'summary_severity.json']
